# Verification — `{{NAME}}` against `{{REVISION}}`

This notebook is the **executed report**, not the source of truth. Every
mathematical claim lives as a test under `tests/`; this page runs them and
shows the evidence, so the trace of *what was exercised* survives in the repo.

Re-run headlessly from the target repo, with its own interpreter:

```
.venv/bin/jupyter nbconvert --to notebook --execute --inplace \
  {{NAME}}/Notebooks/verification.ipynb
```

In [ ]:
import ast
import sys
from pathlib import Path

import numpy as np

ROOT = Path.cwd().parents[1]  # <repo>/{{NAME}}/Notebooks -> <repo>
sys.path.insert(0, str(ROOT / "src"))
SEED = 20260804
rng = np.random.default_rng(SEED)


def provenance_table(package: str) -> None:
    """Show which revision each module was written against (read statically)."""
    for file in sorted((ROOT / "src" / package).rglob("*.py")):
        if file.name == "__init__.py":
            continue
        tree = ast.parse(file.read_text())
        for node in tree.body:
            targets = getattr(node, "targets", [])
            if any(getattr(t, "id", None) == "__provenance__" for t in targets):
                prov = ast.literal_eval(node.value)
                print(f"{file.name:<28} {prov['revision']:<26} §{','.join(prov['sections'])}")


provenance_table("{{NAME}}")

## Levels 1 and 2 — smoke and invariants

Runs the real test suite. A red cell here means the implementation does not
match what the proposal claims. Do not edit the assertion to make it pass.

In [ ]:
import pytest

exit_code = pytest.main(["-q", str(ROOT / "tests")])
assert exit_code == 0, f"test suite failed (pytest exit code {exit_code})"

## Level 3 — synthetic data

Deterministic, fixed seed, cheap. Generate data whose ground truth is known by
construction, then check the implementation recovers it. State the expected
behaviour **before** looking at the output.

In [ ]:
# from {{NAME}}.{{MODULE}} import {{FUNCTION_NAME}}
#
# X = rng.normal(size=(200, 4))
# result = {{FUNCTION_NAME}}(X)
# Save any figure or table under ../Results/ so the evidence is versioned.
